# Relational Generation

Real datasets are rarely a single flat table. Gendantic can generate **multiple related models at once**, keeping foreign keys pointing at real rows so the result has referential integrity.

This notebook covers:
- Declaring primary and foreign keys with `PrimaryKey` / `ForeignKey`
- Generating a whole dataset with `generate_dataset`
- Referential integrity and topological ordering (children generated after parents)
- Relational context: a child's LLM-generated text can be coherent with the parent it references
- Self-references and nullable foreign keys
- Turning the result into pandas DataFrames

## Setup

Like the other notebooks, gendantic uses LiteLLM for the semantic (LLM-generated) fields. Configure your environment:

```bash
export LITELLM_API_BASE="http://localhost:4000"
export LITELLM_API_KEY="your-key"        # if the proxy requires auth
export LITELLM_MODEL="gpt-4o-mini"        # optional, defaults to gpt-4o-mini
```

Or create a `.env` file in your project root (see `.env.example`).

In [ ]:
from typing import Annotated, Optional

from pydantic import BaseModel

from gendantic import (
    generate_dataset,
    PrimaryKey,
    ForeignKey,
    Categorical,
    Normal,
)

## Declaring keys

Two annotations describe the relationships between models:

- **`PrimaryKey()`** marks a field as the model's unique identifier. By default it auto-increments; use `PrimaryKey(strategy="uuid")` for string UUIDs.
- **`ForeignKey(Parent)`** marks a field as a reference to another model's primary key. During generation, gendantic assigns each foreign key a real primary-key value drawn from the parent's generated rows.

Here a `Customer` places many `Order`s, and each `Order` references a `Product`.

In [ ]:
class Customer(BaseModel):
    id: Annotated[int, PrimaryKey()]
    name: str  # LLM-generated
    country: Annotated[str, Categorical(weights={"UK": 0.6, "US": 0.25, "DE": 0.15})]


class Product(BaseModel):
    id: Annotated[int, PrimaryKey()]
    name: str  # LLM-generated
    price: Annotated[float, Normal(mean=40, std=15)]


class Order(BaseModel):
    id: Annotated[int, PrimaryKey()]
    customer_id: Annotated[int, ForeignKey(Customer)]
    product_id: Annotated[int, ForeignKey(Product)]
    quantity: Annotated[int, Categorical(weights={"1": 0.5, "2": 0.3, "3": 0.2})]

## Generating the dataset

Pass a mapping of `{model: count}` to `generate_dataset`. Gendantic sorts the models topologically, so parents (`Customer`, `Product`) are always generated before the children (`Order`) that reference them — the declaration order doesn't matter.

In [ ]:
dataset = await generate_dataset(
    {Customer: 8, Product: 5, Order: 20},
    seed=42,
)

customers = dataset[Customer]
products = dataset[Product]
orders = dataset[Order]

print(f"{len(customers)} customers, {len(products)} products, {len(orders)} orders\n")
for o in orders[:5]:
    print(f"  Order {o.id}: customer={o.customer_id} product={o.product_id} qty={o.quantity}")

## Referential integrity

Every foreign key points at a primary key that actually exists in the dataset.

In [ ]:
customer_ids = {c.id for c in customers}
product_ids = {p.id for p in products}

assert all(o.customer_id in customer_ids for o in orders)
assert all(o.product_id in product_ids for o in orders)
assert len(customer_ids) == len(customers)  # primary keys are unique

print("Referential integrity verified: every order references a real customer and product.")

## Relational context

When a child model has LLM-generated fields, gendantic shows the LLM the *actual* parent rows a record's foreign keys point at. That way the generated text can be coherent with what it references.

For example, a `Review` references a `Product` and generates a `comment`. The LLM sees the referenced product's name, so the comment reads as feedback about *that* product rather than something generic.

In [ ]:
class Review(BaseModel):
    id: Annotated[int, PrimaryKey()]
    product_id: Annotated[int, ForeignKey(Product)]
    rating: Annotated[int, Categorical(weights={"1": 0.1, "2": 0.1, "3": 0.2, "4": 0.3, "5": 0.3})]
    comment: str  # LLM-generated, using the referenced product as context


review_data = await generate_dataset({Product: 4, Review: 8}, seed=7)

products_by_id = {p.id: p for p in review_data[Product]}
for r in review_data[Review]:
    product = products_by_id[r.product_id]
    print(f"  {r.rating}/5 on '{product.name}': {r.comment}")

## Self-references and nullable foreign keys

A foreign key can point back at the same model. You can't name a class inside its own body, so pass its name as a **string forward reference**: `ForeignKey("Employee")`.

Marking the key `nullable=True` lets some rows have no parent — here, top-level employees with `manager_id = None`. `null_probability` controls how often that happens.

In [ ]:
class Employee(BaseModel):
    id: Annotated[int, PrimaryKey()]
    name: str  # LLM-generated
    manager_id: Annotated[
        Optional[int],
        ForeignKey("Employee", nullable=True, null_probability=0.25),
    ] = None


org = await generate_dataset({Employee: 12}, seed=3)
employees = org[Employee]

employee_ids = {e.id for e in employees}
assert all(e.manager_id in employee_ids for e in employees if e.manager_id is not None)

top_level = [e for e in employees if e.manager_id is None]
print(f"{len(employees)} employees, {len(top_level)} with no manager (top level).\n")
for e in employees[:6]:
    reports_to = e.manager_id if e.manager_id is not None else "-"
    print(f"  {e.id}: {e.name} (manager: {reports_to})")

## Exporting to DataFrames

`dataset.to_dataframes()` returns a `{model_name: DataFrame}` mapping (requires the `pandas` extra), ready for analysis or export.

In [ ]:
frames = dataset.to_dataframes()
print("Tables:", {name: df.shape for name, df in frames.items()})
frames["Order"].head()

## Notes

- Not in an async context (a plain script or REPL)? Use `generate_dataset_sync(...)` instead of `await generate_dataset(...)`.
- See `examples/relational_quickstart.py`, `examples/relational_hierarchy.py`, and `examples/relational_llm.py` for runnable scripts covering the same ground.